In [7]:
import os
import logging
import numpy as np
import scipy.io as sio
import time
from obspy import read, UTCDateTime
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# Helper: Convert UTC string to MATLAB datenum
def utc_to_datenum(dt_str):
    dt = datetime.strptime(dt_str, "%Y-%m-%d:%H:%M:%S.%f")
    return dt.toordinal() + 366 + (dt.hour + dt.minute / 60 + dt.second / 3600 + dt.microsecond / 3.6e9) / 24

# Logging config
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def download_with_curl(start, end, network, station_pattern, location_pattern, channel_patterns, username, token, output_path):
    url = (
        "http://service.iris.edu/fdsnws/dataselect/1/queryauth"
        f"?network={network}"
        f"&station={station_pattern}"
        f"&location={location_pattern}"
        f"&channel={','.join(channel_patterns)}"
        f"&starttime={start}"
        f"&endtime={end}"
    )
    curl_cmd = f'curl --digest --user {username}:{token} "{url}" -o "{output_path}"'
    result = os.system(curl_cmd)
    if result != 0:
        logger.warning(f"curl failed for {start} to {end}")

def process_interval(current_date, args):
    (network, station_pattern, location_pattern, channel_patterns,
     username, token, output_folder, output_foldermseed,
     increment_seconds, overlap_seconds) = args

    interval_start = current_date - overlap_seconds
    interval_end = current_date + increment_seconds + overlap_seconds
    date_str = current_date.strftime('%Y-%m-%d-%H-%M-%S')

    year = current_date.strftime('%Y')
    month = current_date.strftime('%m')
    output_dir = os.path.join(output_folder, year, month)
    output_dir_mseed = os.path.join(output_foldermseed, year, month)
    os.makedirs(output_dir_mseed, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    output_mseed = os.path.join(output_dir_mseed, f"{date_str}.mseed")
    output_mat = os.path.join(output_dir, f"{date_str}.mat")

    if os.path.exists(output_mat):
        logger.info(f"Already processed: {output_mat}")
        return

    # Download data
    download_with_curl(
        start=interval_start.strftime("%Y-%m-%dT%H:%M:%S"),
        end=interval_end.strftime("%Y-%m-%dT%H:%M:%S"),
        network=network,
        station_pattern=station_pattern,
        location_pattern=location_pattern,
        channel_patterns=channel_patterns,
        username=username,
        token=token,
        output_path=output_mseed
    )

    # Read and save
    try:
        stream = read(output_mseed)
        trace_dict = {
            'network': [], 'station': [], 'location': [], 'channel': [],
            'sensitivity': [], 'sensitivityFrequency': [], 'data': [],
            'sampleCount': [], 'sampleRate': [], 'startTime': [], 'endTime': []
        }

        for trace in stream:
            trace_dict['network'].append(trace.stats.network)
            trace_dict['station'].append(trace.stats.station)
            trace_dict['location'].append(trace.stats.location)
            trace_dict['channel'].append(trace.stats.channel)
            trace_dict['sensitivity'].append(np.nan)
            trace_dict['sensitivityFrequency'].append(np.nan)
            trace_dict['data'].append(trace.data.astype(np.float64))
            trace_dict['sampleCount'].append(int(trace.stats.npts))
            trace_dict['sampleRate'].append(float(trace.stats.sampling_rate))
            trace_dict['startTime'].append(trace.stats.starttime.strftime("%Y-%m-%d:%H:%M:%S.%f"))
            trace_dict['endTime'].append(trace.stats.endtime.strftime("%Y-%m-%d:%H:%M:%S.%f"))

        if trace_dict['network']:
            num_traces = len(trace_dict['network'])
            dtype = [
                ('network', 'O'), ('station', 'O'), ('location', 'O'), ('channel', 'O'),
                ('sensitivity', 'f8'), ('sensitivityFrequency', 'f8'), ('data', 'O'),
                ('sampleCount', 'i4'), ('sampleRate', 'f8'),
                ('startTime', 'f8'), ('endTime', 'f8')
            ]

            trace_struct = np.zeros(num_traces, dtype=dtype)
            for i in range(num_traces):
                trace_struct[i] = (
                    trace_dict['network'][i],
                    trace_dict['station'][i],
                    trace_dict['location'][i],
                    trace_dict['channel'][i],
                    trace_dict['sensitivity'][i],
                    trace_dict['sensitivityFrequency'][i],
                    trace_dict['data'][i],
                    trace_dict['sampleCount'][i],
                    trace_dict['sampleRate'][i],
                    utc_to_datenum(trace_dict['startTime'][i]),
                    utc_to_datenum(trace_dict['endTime'][i])
                )

            sio.savemat(output_mat, {'trace': trace_struct}, format='5', do_compression=True)
            file_size = os.path.getsize(output_mat) / 1024
            logger.info(f"✅ Saved {output_mat} ({file_size:.2f} KB), {num_traces} traces")

    except Exception as e:
        logger.error(f"❌ Failed to read or process {output_mseed}: {str(e)}")

def download_seismic_data_with_curl(
    start_datetime="2024-03-04T04:00:00",
    end_datetime="2024-10-05T18:00:00",
    network="2F",
    station_pattern="AX*",
    location_pattern="*",
    channel_patterns=["HH?", "EL?"],
    output_folder="data",
    output_foldermseed="datamseed",
    username="mczhang8@uw.edu",
    token="RxL42LH6XTNFybHb",
    increment_hours=1,
    overlap_seconds=15,
    max_workers=4
):
    try:
        start = UTCDateTime(start_datetime)
        end = UTCDateTime(end_datetime)
        logger.info(f"Processing range: {start_datetime} to {end_datetime}")
    except Exception as e:
        logger.error(f"Invalid datetime format: {str(e)}")
        return

    increment_seconds = increment_hours * 3600
    overlap_seconds = float(overlap_seconds)

    # Create a list of times to process
    date_list = []
    current_date = start
    while current_date <= end:
        date_list.append(current_date)
        current_date += increment_seconds

    args = (
        network, station_pattern, location_pattern, channel_patterns,
        username, token, output_folder, output_foldermseed,
        increment_seconds, overlap_seconds
    )

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_interval, dt, args) for dt in date_list]
        for f in as_completed(futures):
            pass
    elapsed = time.time() - t0
    logger.info(f"🎯 Total time: {elapsed:.2f} seconds")

# Run the script
if __name__ == "__main__":
    download_seismic_data_with_curl()


2025-05-22 10:36:10,253 - INFO - Processing range: 2024-03-04T04:00:00 to 2024-10-05T18:00:00
2025-05-22 10:36:10,302 - INFO - Already processed: data/2024/03/2024-03-04-05-00-00.mat
2025-05-22 10:36:10,313 - INFO - Already processed: data/2024/03/2024-03-04-04-00-00.mat
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0   669    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 25.3M    0 25.3M    0     0  6247k      0 --:--:--  0:00:04 --:--:-- 7275k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0   669    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 25.7M    0 25.7M    0     0  8371k      0 --:--:--  0:00:03 --:--:-- 9897k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                